In [1]:
print("NLP501 - Exercise 3: Neural Networks for NLP")
print("=" * 60)
print("Part A: Sentiment Analysis with RNN/LSTM")
print("Part B: Named Entity Recognition with BiLSTM")
print("Part C: Siamese Network for Question Similarity")

NLP501 - Exercise 3: Neural Networks for NLP
Part A: Sentiment Analysis with RNN/LSTM
Part B: Named Entity Recognition with BiLSTM
Part C: Siamese Network for Question Similarity


In [2]:
# ============================================================
# PART A: SENTIMENT ANALYSIS WITH RNN/LSTM
# ============================================================
# Install required packages
import subprocess
import sys

def install(package):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

from collections import Counter
import re
import json
import os
import time
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.10.0+cu128


In [3]:
# ============================================================
# PART A - Step 1: Load IMDB Dataset & Preprocessing
# ============================================================

# Load IMDB dataset from Keras (no internet needed - built-in)
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Hyperparameters
VOCAB_SIZE = 20000      # Top 20k most frequent words
MAX_LEN = 256           # Max sequence length
EMBED_DIM = 128         # Embedding dimension
HIDDEN_DIM = 256        # LSTM hidden size
NUM_LAYERS = 2          # Number of LSTM layers
DROPOUT = 0.3
BATCH_SIZE = 64
EPOCHS_A = 10
LR_A = 0.001

print("Loading IMDB dataset...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f"Training samples: {len(x_train)}")
print(f"Test samples:     {len(x_test)}")
print(f"Sample sequence length: {len(x_train[0])}")
print(f"Label distribution (train): 0={sum(y_train==0)}, 1={sum(y_train==1)}")

# Pad sequences
x_train_pad = pad_sequences(x_train, maxlen=MAX_LEN, padding='post', truncating='post')
x_test_pad  = pad_sequences(x_test,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f"\nPadded train shape: {x_train_pad.shape}")
print(f"Padded test shape:  {x_test_pad.shape}")

# Split train into train/val
x_tr, x_val, y_tr, y_val = train_test_split(
        x_train_pad, y_train, test_size=0.1, random_state=SEED, stratify=y_train
)
print(f"\nFinal split — Train: {len(x_tr)}, Val: {len(x_val)}, Test: {len(x_test_pad)}")

2026-06-05 01:54:08.548539: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780624448.787500      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780624448.853664      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780624449.390898      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780624449.390940      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780624449.390943      58 computation_placer.cc:177] computation placer alr

Loading IMDB dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Test samples:     25000
Sample sequence length: 218
Label distribution (train): 0=12500, 1=12500

Padded train shape: (25000, 256)
Padded test shape:  (25000, 256)

Final split — Train: 22500, Val: 2500, Test: 25000


In [4]:
# ============================================================
# PART A - Step 2: Dataset, DataLoader & LSTM Model
# ============================================================

class IMDBDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels    = torch.tensor(labels,    dtype=torch.float)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_dataset = IMDBDataset(x_tr,        y_tr)
val_dataset   = IMDBDataset(x_val,       y_val)
test_dataset  = IMDBDataset(x_test_pad,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

# -------- LSTM Sentiment Model --------
class LSTMSentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))           # (B, L, E)
        output, (hidden, _) = self.lstm(embedded)            # hidden: (layers, B, H)
        hidden_last = self.dropout(hidden[-1])               # (B, H)
        logit = self.fc(hidden_last).squeeze(1)              # (B,)
        return logit

model_a = LSTMSentimentModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
print(model_a)
total_params = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

Train batches: 352, Val batches: 40, Test batches: 391
LSTMSentimentModel(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)

Trainable parameters: 3,481,857


In [5]:
# ============================================================
# PART A - Step 3: Training Loop with Loss & Accuracy Curves
# ============================================================

criterion_a = nn.BCEWithLogitsLoss()
optimizer_a = optim.Adam(model_a.parameters(), lr=LR_A, weight_decay=1e-5)
scheduler_a = optim.lr_scheduler.ReduceLROnPlateau(optimizer_a, 'min', patience=2, factor=0.5)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for seqs, labels in loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(seqs)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total += len(labels)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for seqs, labels in loader:
            seqs, labels = seqs.to(device), labels.to(device)
            logits = model(seqs)
            loss = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            preds = (torch.sigmoid(logits) >= 0.5).long()
            correct += (preds == labels.long()).sum().item()
            total += len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().long().numpy())
    return total_loss / total, correct / total, all_preds, all_labels

# Training
history_a = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
best_model_state = None

print("Training LSTM Sentiment Model...")
print(f"{'Epoch':>5} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>9} {'Val Acc':>8}")
print("-" * 50)

for epoch in range(1, EPOCHS_A + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model_a, train_loader, criterion_a, optimizer_a, DEVICE)
    vl_loss, vl_acc, _, _ = eval_epoch(model_a, val_loader, criterion_a, DEVICE)
    scheduler_a.step(vl_loss)
    history_a['train_loss'].append(tr_loss)
    history_a['val_loss'].append(vl_loss)
    history_a['train_acc'].append(tr_acc)
    history_a['val_acc'].append(vl_acc)
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_model_state = {k: v.cpu().clone() for k, v in model_a.state_dict().items()}
    elapsed = time.time() - t0
    print(f"{epoch:>5} {tr_loss:>11.4f} {tr_acc:>10.4f} {vl_loss:>9.4f} {vl_acc:>8.4f}  ({elapsed:.1f}s)")

print("\nTraining complete!")

Training LSTM Sentiment Model...
Epoch  Train Loss  Train Acc  Val Loss  Val Acc
--------------------------------------------------
    1      0.6931     0.5069    0.6917   0.5184  (15.6s)
    2      0.6904     0.5159    0.6872   0.5248  (15.0s)
    3      0.6866     0.5321    0.6834   0.5408  (15.1s)
    4      0.6736     0.5729    0.6877   0.5332  (15.1s)
    5      0.6477     0.6187    0.6875   0.5332  (15.6s)
    6      0.6107     0.6633    0.5748   0.7072  (15.9s)
    7      0.5444     0.7448    0.5408   0.7580  (16.2s)
    8      0.4614     0.7994    0.4470   0.8112  (16.7s)
    9      0.4157     0.8253    0.4538   0.8028  (17.1s)
   10      0.4235     0.8296    0.4751   0.7780  (16.8s)

Training complete!


In [6]:
# ============================================================
# PART A - Step 4: Training Curves & Test Evaluation
# ============================================================

# Plot training curves
epochs_range = range(1, EPOCHS_A + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Part A: LSTM Sentiment Analysis - Training Curves", fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history_a['train_loss'], 'b-o', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history_a['val_loss'],   'r-s', label='Val Loss',   linewidth=2)
axes[0].set_title('Loss Curves'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history_a['train_acc'], 'b-o', label='Train Accuracy', linewidth=2)
axes[1].plot(epochs_range, history_a['val_acc'],   'r-s', label='Val Accuracy',   linewidth=2)
axes[1].set_title('Accuracy Curves'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0.5, 1.0])

plt.tight_layout()
plt.savefig('/kaggle/working/partA_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved!")

# Load best model and evaluate on test set
if best_model_state:
    model_a.load_state_dict(best_model_state)
    model_a = model_a.to(DEVICE)

test_loss, test_acc, test_preds, test_labels = eval_epoch(model_a, test_loader, criterion_a, DEVICE)

print(f"\n{'='*60}")
print(f"PART A TEST RESULTS")
print(f"{'='*60}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=['Negative', 'Positive']))

# Save model weights
torch.save(model_a.state_dict(), '/kaggle/working/partA_lstm_sentiment.pth')
print("Model saved to /kaggle/working/partA_lstm_sentiment.pth")

Training curves saved!

PART A TEST RESULTS
Test Loss:     0.4781
Test Accuracy: 0.7980 (79.80%)

Classification Report:
              precision    recall  f1-score   support

    Negative       0.80      0.80      0.80     12500
    Positive       0.80      0.80      0.80     12500

    accuracy                           0.80     25000
   macro avg       0.80      0.80      0.80     25000
weighted avg       0.80      0.80      0.80     25000

Model saved to /kaggle/working/partA_lstm_sentiment.pth


In [7]:
# ============================================================
# PART B: NAMED ENTITY RECOGNITION WITH BiLSTM
# ============================================================

import random
random.seed(SEED)

CONLL_NER_TAGS = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC",
]

def parse_conll(filepath):
    sentences = []
    words, tags = [], []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('-DOCSTART-') or line == '':
                if words:
                    sentences.append((words[:], tags[:]))
                    words, tags = [], []
            else:
                parts = line.split()
                if len(parts) >= 4:
                    words.append(parts[0])
                    tags.append(parts[3])
    if words:
        sentences.append((words, tags))
    return sentences

def load_hf_conll():
    from datasets import load_dataset

    # The original "conll2003" loader is script-based and fails on recent
    # Kaggle/datasets versions. This mirror is stored as Parquet.
    ds = load_dataset("lhoestq/conll2003")

    feature = ds['train'].features.get('ner_tags')
    if feature is not None and hasattr(feature, 'feature') and hasattr(feature.feature, 'names'):
        tag_names = list(feature.feature.names)
    else:
        tag_names = CONLL_NER_TAGS

    def convert(split):
        converted = []
        for sample in split:
            ner_tags = sample['ner_tags']
            tags = [tag_names[t] if isinstance(t, int) else str(t) for t in ner_tags]
            converted.append((sample['tokens'], tags))
        return converted

    return convert(ds['train']), convert(ds['validation']), convert(ds['test'])

# 1) Try local Kaggle dataset paths
CONLL_CANDIDATES = [
    ('/kaggle/input/conll2003/train.txt',           '/kaggle/input/conll2003/valid.txt',           '/kaggle/input/conll2003/test.txt'),
    ('/kaggle/input/conll-2003/train.txt',          '/kaggle/input/conll-2003/valid.txt',          '/kaggle/input/conll-2003/test.txt'),
    ('/kaggle/input/conll-2003-dataset/train.txt',  '/kaggle/input/conll-2003-dataset/valid.txt',  '/kaggle/input/conll-2003-dataset/test.txt'),
    ('/kaggle/input/conll2003-english/train.txt',   '/kaggle/input/conll2003-english/valid.txt',   '/kaggle/input/conll2003-english/test.txt'),
]

train_b = val_b = test_b = None
for tr, vl, te in CONLL_CANDIDATES:
    if os.path.exists(tr):
        print(f"Loading CoNLL-2003 from local path: {tr}")
        train_b = parse_conll(tr)
        val_b   = parse_conll(vl) if os.path.exists(vl) else []
        test_b  = parse_conll(te) if os.path.exists(te) else []
        break

# 2) Fall back to HuggingFace datasets (requires Kaggle internet enabled)
if train_b is None:
    print("Local CoNLL-2003 not found - loading from HuggingFace Parquet mirror...")
    print("(If this fails, go to Kaggle Settings > Internet > Turn ON)")
    try:
        train_b, val_b, test_b = load_hf_conll()
        print("Loaded from HuggingFace successfully.")
    except Exception as e:
        raise RuntimeError(
            f"Could not load CoNLL-2003: {e}\n"
            "Options:\n"
            "  A) Enable Internet in Kaggle notebook Settings, then re-run.\n"
            "  B) Add a CoNLL-2003 Kaggle dataset with train/valid/test .txt files."
        ) from e

print(f"Train: {len(train_b)}, Val: {len(val_b)}, Test: {len(test_b)} sentences")
print(f"\nExample sentence:")
ws, ts = train_b[0]
for w, t in zip(ws, ts):
    print(f"  {w:15s} {t}")


Local CoNLL-2003 not found - loading from HuggingFace Parquet mirror...
(If this fails, go to Kaggle Settings > Internet > Turn ON)


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Loaded from HuggingFace successfully.
Train: 14041, Val: 3250, Test: 3453 sentences

Example sentence:
  EU              B-ORG
  rejects         O
  German          B-MISC
  call            O
  to              O
  boycott         O
  British         B-MISC
  lamb            O
  .               O


In [8]:
# ============================================================
# PART B - Step 2: Vocabulary Building & BIO Dataset
# ============================================================

# Build vocabularies from training data
all_words = [w for ws, _ in train_b for w in ws]
tag_set   = sorted(set(t for _, ts in (train_b + val_b + test_b) for t in ts))

word_counter = Counter(all_words)

# Vocab: PAD=0, UNK=1
word2idx = {"<PAD>": 0, "<UNK>": 1}
for w, _ in word_counter.most_common():
    word2idx[w] = len(word2idx)
idx2word = {v: k for k, v in word2idx.items()}

tag2idx = {t: i for i, t in enumerate(tag_set)}
idx2tag  = {i: t for t, i in tag2idx.items()}

VOCAB_B  = len(word2idx)
NUM_TAGS = len(tag2idx)
PAD_TAG  = -100   # PyTorch ignore_index for padding only; "O" remains a normal label

print(f"Vocabulary size: {VOCAB_B}")
print(f"Tag set ({NUM_TAGS}): {tag_set}")
print(f"tag2idx: {tag2idx}")
print(f"\nSplit - Train: {len(train_b)}, Val: {len(val_b)}, Test: {len(test_b)}")

# NER Dataset
class NERDataset(Dataset):
    def __init__(self, data, word2idx, tag2idx, max_len=50):
        self.samples = []
        for words, tags in data:
            wids = [word2idx.get(w, 1) for w in words[:max_len]]
            tids = [tag2idx[t] for t in tags[:max_len]]
            self.samples.append((wids, tids))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

def ner_collate(batch):
    words_list, tags_list = zip(*batch)
    max_l = max(len(w) for w in words_list)
    padded_words = torch.zeros(len(batch), max_l, dtype=torch.long)
    padded_tags  = torch.full((len(batch), max_l), PAD_TAG, dtype=torch.long)
    mask = torch.zeros(len(batch), max_l, dtype=torch.bool)
    for i, (ws, ts) in enumerate(zip(words_list, tags_list)):
        padded_words[i, :len(ws)] = torch.tensor(ws)
        padded_tags[i,  :len(ts)] = torch.tensor(ts)
        mask[i, :len(ws)] = True
    return padded_words, padded_tags, mask

train_ner_ds = NERDataset(train_b, word2idx, tag2idx)
val_ner_ds   = NERDataset(val_b,   word2idx, tag2idx)
test_ner_ds  = NERDataset(test_b,  word2idx, tag2idx)

BATCH_B = 32
train_ner_dl = DataLoader(train_ner_ds, batch_size=BATCH_B, shuffle=True,  collate_fn=ner_collate)
val_ner_dl   = DataLoader(val_ner_ds,   batch_size=BATCH_B, shuffle=False, collate_fn=ner_collate)
test_ner_dl  = DataLoader(test_ner_ds,  batch_size=BATCH_B, shuffle=False, collate_fn=ner_collate)
print(f"Train NER batches: {len(train_ner_dl)}")

Vocabulary size: 23625
Tag set (9): ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
tag2idx: {'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}

Split - Train: 14041, Val: 3250, Test: 3453
Train NER batches: 439


In [9]:
# ============================================================
# PART B - Step 3: BiLSTM NER Model & Training
# ============================================================

from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class BiLSTMNER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_tags, dropout=0.3):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bilstm     = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_tags)  # *2 for bidirectional

    def forward(self, x):
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()
        emb = self.dropout(self.embedding(x))                 # (B, L, E)
        packed = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.bilstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))
        out = self.dropout(out)
        logits = self.classifier(out)                         # (B, L, num_tags)
        return logits

def extract_entities(tag_seq, idx2tag):
    entities = []
    cur_type = None
    start = None
    for i, tid in enumerate(tag_seq):
        tag = idx2tag[int(tid)]
        if tag.startswith("B-"):
            if cur_type is not None:
                entities.append((cur_type, start, i - 1))
            cur_type = tag[2:]
            start = i
        elif tag.startswith("I-"):
            ent_type = tag[2:]
            if cur_type != ent_type:
                if cur_type is not None:
                    entities.append((cur_type, start, i - 1))
                cur_type = ent_type
                start = i
        else:
            if cur_type is not None:
                entities.append((cur_type, start, i - 1))
            cur_type = None
            start = None
    if cur_type is not None:
        entities.append((cur_type, start, len(tag_seq) - 1))
    return entities

def entity_f1(pred_seqs, true_seqs, idx2tag):
    entity_types = ["PER", "LOC", "ORG", "MISC"]
    results = {}
    macro_f1 = 0.0
    n_types = 0
    for etype in entity_types:
        tp = fp = fn = 0
        for ps, ts in zip(pred_seqs, true_seqs):
            pred_ents = set((t, s, e) for t, s, e in extract_entities(ps, idx2tag) if t == etype)
            true_ents = set((t, s, e) for t, s, e in extract_entities(ts, idx2tag) if t == etype)
            tp += len(pred_ents & true_ents)
            fp += len(pred_ents - true_ents)
            fn += len(true_ents - pred_ents)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        support = tp + fn
        results[etype] = {"precision": precision, "recall": recall, "f1": f1, "support": support}
        if support > 0:
            macro_f1 += f1
            n_types += 1
    return results, macro_f1 / max(n_types, 1)

EMBED_B  = 64
HIDDEN_B = 128
LAYERS_B = 2
LR_B     = 0.001
EPOCHS_B = 15

model_b = BiLSTMNER(VOCAB_B, EMBED_B, HIDDEN_B, LAYERS_B, NUM_TAGS).to(DEVICE)
print(model_b)
print(f"Trainable params: {sum(p.numel() for p in model_b.parameters() if p.requires_grad):,}")

criterion_b = nn.CrossEntropyLoss(ignore_index=PAD_TAG)
optimizer_b = optim.Adam(model_b.parameters(), lr=LR_B)
scheduler_b = optim.lr_scheduler.StepLR(optimizer_b, step_size=5, gamma=0.5)

def train_ner_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for words, tags, mask in loader:
        words, tags, mask = words.to(device), tags.to(device), mask.to(device)
        optimizer.zero_grad()
        logits = model(words)                                         # (B, L, T)
        B, L, T = logits.shape
        loss = criterion(logits.reshape(B*L, T), tags.reshape(B*L))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += ((preds == tags) & mask).sum().item()
        total   += mask.sum().item()
    return total_loss / len(loader), correct / total

def eval_ner_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds_flat, all_tags_flat = [], []
    pred_seqs, true_seqs = [], []
    with torch.no_grad():
        for words, tags, mask in loader:
            words, tags, mask = words.to(device), tags.to(device), mask.to(device)
            logits = model(words)
            B, L, T = logits.shape
            loss = criterion(logits.reshape(B*L, T), tags.reshape(B*L))
            total_loss += loss.item()
            preds = logits.argmax(dim=-1)
            correct += ((preds == tags) & mask).sum().item()
            total   += mask.sum().item()
            for i in range(B):
                l = int(mask[i].sum().item())
                pred_seq = preds[i, :l].cpu().tolist()
                true_seq = tags[i,  :l].cpu().tolist()
                pred_seqs.append(pred_seq)
                true_seqs.append(true_seq)
                all_preds_flat.extend(pred_seq)
                all_tags_flat.extend(true_seq)
    return total_loss / len(loader), correct / total, all_preds_flat, all_tags_flat, pred_seqs, true_seqs

# Train
history_b = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_entity_f1': []}
best_val_entity_f1_b = -1.0
best_epoch_b = 0
best_model_state_b = None

print("Training BiLSTM NER model...")
print(f"{'Epoch':>5} {'Train Loss':>11} {'Token Acc':>10} {'Val Loss':>9} {'Val Acc':>8} {'Val EntF1':>10}")
print("-" * 62)

for epoch in range(1, EPOCHS_B + 1):
    tr_loss, tr_acc = train_ner_epoch(model_b, train_ner_dl, criterion_b, optimizer_b, DEVICE)
    vl_loss, vl_acc, _, _, val_pred_seqs, val_true_seqs = eval_ner_epoch(model_b, val_ner_dl, criterion_b, DEVICE)
    _, val_entity_f1 = entity_f1(val_pred_seqs, val_true_seqs, idx2tag)
    scheduler_b.step()
    history_b['train_loss'].append(tr_loss);     history_b['val_loss'].append(vl_loss)
    history_b['train_acc'].append(tr_acc);       history_b['val_acc'].append(vl_acc)
    history_b['val_entity_f1'].append(val_entity_f1)
    if val_entity_f1 > best_val_entity_f1_b:
        best_val_entity_f1_b = val_entity_f1
        best_epoch_b = epoch
        best_model_state_b = {k: v.cpu().clone() for k, v in model_b.state_dict().items()}
    print(f"{epoch:>5} {tr_loss:>11.4f} {tr_acc:>10.4f} {vl_loss:>9.4f} {vl_acc:>8.4f} {val_entity_f1:>10.4f}")

if best_model_state_b is not None:
    model_b.load_state_dict(best_model_state_b)
    model_b = model_b.to(DEVICE)

print(f"\nNER Training complete! Best Val Entity F1: {best_val_entity_f1_b:.4f} at epoch {best_epoch_b}")

BiLSTMNER(
  (embedding): Embedding(23625, 64, padding_idx=0)
  (bilstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=9, bias=True)
)
Trainable params: 2,108,233
Training BiLSTM NER model...
Epoch  Train Loss  Token Acc  Val Loss  Val Acc  Val EntF1
--------------------------------------------------------------
    1      0.6357     0.8472    0.4378   0.8748     0.3073
    2      0.3880     0.8882    0.3066   0.9083     0.5032
    3      0.2952     0.9125    0.2519   0.9255     0.6188
    4      0.2384     0.9286    0.2147   0.9362     0.6670
    5      0.2009     0.9385    0.1950   0.9410     0.6951
    6      0.1666     0.9489    0.1848   0.9439     0.7086
    7      0.1516     0.9524    0.1760   0.9464     0.7135
    8      0.1407     0.9559    0.1738   0.9485     0.7320
    9      0.1319     0.9583    0.1682   0.9501     0.7431
   10      0.1208     0.9

In [10]:
# ============================================================
# PART B - Step 4: NER Evaluation with Entity-level F1
# ============================================================

# Evaluate on test set with the best validation-entity-F1 checkpoint loaded in Cell 9
_, test_tok_acc, test_preds_b, test_tags_b, pred_seqs_b, true_seqs_b = eval_ner_epoch(
    model_b, test_ner_dl, criterion_b, DEVICE
)

ner_results, test_macro_f1_b = entity_f1(pred_seqs_b, true_seqs_b, idx2tag)

print(f"\n{'='*60}")
print(f"PART B NER RESULTS - Entity-level F1")
print(f"{'='*60}")
print(f"Best Val Entity F1:      {best_val_entity_f1_b:.4f} (epoch {best_epoch_b})")
print(f"Token-level Test Accuracy: {test_tok_acc:.4f}")
print(f"Test Entity Macro F1:      {test_macro_f1_b:.4f}")
print(f"\n{'Entity':<8} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Support':>9}")
print("-" * 45)
for etype, m in ner_results.items():
    print(f"{etype:<8} {m['precision']:>10.4f} {m['recall']:>8.4f} {m['f1']:>8.4f} {m['support']:>9d}")
print("-" * 45)
print(f"{'Macro':<8} {' ':>10} {' ':>8} {test_macro_f1_b:>8.4f}")

# Training curves for NER
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle("Part B: BiLSTM NER - Training Curves", fontsize=14, fontweight='bold')
epochs_b = range(1, EPOCHS_B+1)
axes2[0].plot(epochs_b, history_b['train_loss'], 'b-o', label='Train Loss', linewidth=2)
axes2[0].plot(epochs_b, history_b['val_loss'],   'r-s', label='Val Loss',   linewidth=2)
axes2[0].set_title('NER Loss'); axes2[0].set_xlabel('Epoch'); axes2[0].legend(); axes2[0].grid(True, alpha=0.3)
axes2[1].plot(epochs_b, history_b['train_acc'], 'b-o', label='Train Token Acc', linewidth=2)
axes2[1].plot(epochs_b, history_b['val_acc'],   'r-s', label='Val Token Acc',   linewidth=2)
axes2[1].set_title('NER Token Accuracy'); axes2[1].set_xlabel('Epoch'); axes2[1].legend(); axes2[1].grid(True, alpha=0.3)
axes2[2].plot(epochs_b, history_b['val_entity_f1'], 'g-o', label='Val Entity F1', linewidth=2)
axes2[2].axvline(best_epoch_b, color='k', linestyle='--', linewidth=1, label=f'Best epoch {best_epoch_b}')
axes2[2].set_title('NER Entity F1'); axes2[2].set_xlabel('Epoch'); axes2[2].legend(); axes2[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/partB_ner_curves.png', dpi=150, bbox_inches='tight')
plt.show()

torch.save(model_b.state_dict(), '/kaggle/working/partB_bilstm_ner.pth')
print("\nNER model saved!")


PART B NER RESULTS - Entity-level F1
Best Val Entity F1:      0.7623 (epoch 15)
Token-level Test Accuracy: 0.9379
Test Entity Macro F1:      0.6890

Entity    Precision   Recall       F1   Support
---------------------------------------------
PER          0.6654   0.7487   0.7046      1580
LOC          0.8485   0.6661   0.7463      1656
ORG          0.7121   0.6369   0.6724      1658
MISC         0.6651   0.6034   0.6328       701
---------------------------------------------
Macro                          0.6890

NER model saved!


In [11]:
# ============================================================
# PART C: SIAMESE NETWORK FOR QUESTION SIMILARITY
# ============================================================

import itertools

def load_hf_quora(n_samples=50000, seed=42):
    from datasets import load_dataset
    ds = load_dataset("glue", "qqp", split="train")
    ds = ds.shuffle(seed=seed).select(range(min(n_samples, len(ds))))
    pairs = [(r['question1'], r['question2'], r['label']) for r in ds]
    return pairs

# 1) Try local Kaggle dataset paths
QUORA_CANDIDATES = [
    '/kaggle/input/quora-question-pairs/train.csv.zip',
    '/kaggle/input/quora-question-pairs/train.csv',
    '/kaggle/input/quora-duplicate-questions/train.csv',
    '/kaggle/input/quora-nlp/train.csv',
]

augmented = None
for p in QUORA_CANDIDATES:
    if os.path.exists(p):
        print(f"Loading Quora QP from local path: {p}")
        df = pd.read_csv(p, nrows=60000)
        df = df[['question1', 'question2', 'is_duplicate']].dropna()
        df['is_duplicate'] = df['is_duplicate'].astype(int)
        pos_df = df[df['is_duplicate'] == 1]
        neg_df = df[df['is_duplicate'] == 0]
        min_len = min(len(pos_df), len(neg_df), 25000)
        balanced = pd.concat([
            pos_df.sample(min_len, random_state=SEED),
            neg_df.sample(min_len, random_state=SEED)
        ]).sample(frac=1, random_state=SEED).reset_index(drop=True)
        augmented = list(zip(
            balanced['question1'].tolist(),
            balanced['question2'].tolist(),
            balanced['is_duplicate'].tolist()
        ))
        break

# 2) Fall back to HuggingFace GLUE/QQP (requires Kaggle internet enabled)
if augmented is None:
    print("Local Quora dataset not found — loading from HuggingFace (GLUE/QQP)...")
    print("(If this fails, go to Kaggle Settings → Internet → Turn ON)")
    try:
        raw_pairs = load_hf_quora(n_samples=50000, seed=SEED)
        # Balance positive/negative
        pos = [(q1, q2, l) for q1, q2, l in raw_pairs if l == 1]
        neg = [(q1, q2, l) for q1, q2, l in raw_pairs if l == 0]
        min_len = min(len(pos), len(neg), 25000)
        import random as _rnd
        _rnd.seed(SEED)
        augmented = _rnd.sample(pos, min_len) + _rnd.sample(neg, min_len)
        _rnd.shuffle(augmented)
        print("Loaded from HuggingFace successfully.")
    except Exception as e:
        raise RuntimeError(
            f"Could not load Quora QP: {e}\n"
            "Options:\n"
            "  A) Enable Internet in Kaggle notebook Settings, then re-run.\n"
            "  B) Add dataset 'quora/question-pairs-dataset' via Kaggle > Add Data."
        ) from e

print(f"Total pairs: {len(augmented)}")
print(f"Positive: {sum(1 for _,_,l in augmented if l==1)}")
print(f"Negative: {sum(1 for _,_,l in augmented if l==0)}")
print(f"\nSample positive: {next((q1, q2) for q1, q2, l in augmented if l==1)}")
print(f"Sample negative: {next((q1, q2) for q1, q2, l in augmented if l==0)}")

Local Quora dataset not found — loading from HuggingFace (GLUE/QQP)...
(If this fails, go to Kaggle Settings → Internet → Turn ON)


README.md: 0.00B [00:00, ?B/s]

qqp/train-00000-of-00001.parquet:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

qqp/validation-00000-of-00001.parquet:   0%|          | 0.00/3.73M [00:00<?, ?B/s]

qqp/test-00000-of-00001.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]

Loaded from HuggingFace successfully.
Total pairs: 36890
Positive: 18445
Negative: 18445

Sample positive: ('Who is the best doctor/dermatologist for hair fall treatment in Bangalore?', 'How can I find best doctor for hair fall treatment in Bangalore?')
Sample negative: ('Why is everybody so crazy about Star Wars?', 'Is Star Wars overrated?')


In [12]:
# ============================================================
# PART C - Step 2: Tokenization & Siamese Dataset
# ============================================================

def question_tokens(q):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", str(q).lower())

# Build vocabulary from question pairs with the same tokenizer used by the dataset
all_q_words = []
for q1, q2, _ in augmented:
    all_q_words.extend(question_tokens(q1))
    all_q_words.extend(question_tokens(q2))

q_word_counter = Counter(all_q_words)
q_word2idx = {"<PAD>": 0, "<UNK>": 1}
for w, cnt in q_word_counter.most_common():
    if cnt >= 1:
        q_word2idx[w] = len(q_word2idx)
VOCAB_C = len(q_word2idx)
print(f"Question vocab size: {VOCAB_C}")

MAX_Q_LEN = 20

def tokenize_question(q, word2idx, max_len):
    words = question_tokens(q)[:max_len]
    ids = [word2idx.get(w, 1) for w in words]
    if len(ids) < max_len:
        ids = ids + [0] * (max_len - len(ids))
    return ids[:max_len]

class SiameseDataset(Dataset):
    def __init__(self, pairs, word2idx, max_len=MAX_Q_LEN):
        self.data = []
        for q1, q2, label in pairs:
            t1 = tokenize_question(q1, word2idx, max_len)
            t2 = tokenize_question(q2, word2idx, max_len)
            self.data.append((
                torch.tensor(t1, dtype=torch.long),
                torch.tensor(t2, dtype=torch.long),
                torch.tensor(label, dtype=torch.float)
            ))
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# Split dataset
n_c = len(augmented)
n_tr_c = int(0.7*n_c); n_vl_c = int(0.15*n_c)
train_c = augmented[:n_tr_c]
val_c   = augmented[n_tr_c:n_tr_c+n_vl_c]
test_c  = augmented[n_tr_c+n_vl_c:]

BATCH_C = 32
train_c_ds = SiameseDataset(train_c, q_word2idx)
val_c_ds   = SiameseDataset(val_c,   q_word2idx)
test_c_ds  = SiameseDataset(test_c,  q_word2idx)

train_c_dl = DataLoader(train_c_ds, batch_size=BATCH_C, shuffle=True)
val_c_dl   = DataLoader(val_c_ds,   batch_size=BATCH_C, shuffle=False)
test_c_dl  = DataLoader(test_c_ds,  batch_size=BATCH_C, shuffle=False)

print(f"Split - Train: {len(train_c)}, Val: {len(val_c)}, Test: {len(test_c)}")

Question vocab size: 27576
Split - Train: 25823, Val: 5533, Test: 5534


In [13]:
# ============================================================
# PART C - Step 3: Siamese Network Architecture & Training
# ============================================================

from sklearn.metrics import f1_score, roc_auc_score
from torch.nn.utils.rnn import pack_padded_sequence

class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()
        emb = self.dropout(self.embedding(x))
        packed = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.dropout(h[-1])   # (B, H)

class SiameseNet(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.encoder = LSTMEncoder(vocab_size, embed_dim, hidden_dim, num_layers, dropout)
        proj_dim = hidden_dim // 2
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
    def encode(self, x):
        return self.projection(self.encoder(x))
    def forward(self, q1, q2):
        z1 = self.encode(q1)
        z2 = self.encode(q2)
        features = torch.cat([z1, z2, torch.abs(z1 - z2), z1 * z2], dim=1)
        logits = self.classifier(features).squeeze(1)
        return logits

EMBED_C=64; HIDDEN_C=128; LAYERS_C=1; LR_C=0.001; EPOCHS_C=20
model_c = SiameseNet(VOCAB_C, EMBED_C, HIDDEN_C, LAYERS_C).to(DEVICE)
print(model_c)
print(f"Params: {sum(p.numel() for p in model_c.parameters() if p.requires_grad):,}")

criterion_c = nn.BCEWithLogitsLoss()
optimizer_c = optim.Adam(model_c.parameters(), lr=LR_C, weight_decay=1e-4)
scheduler_c = optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=EPOCHS_C)

def binary_accuracy_from_probs(probs, labels, threshold=0.5):
    preds = (np.asarray(probs) >= threshold).astype(int)
    labels = np.asarray(labels).astype(int)
    return (preds == labels).mean()

def tune_threshold(probs, labels):
    labels_arr = np.asarray(labels).astype(int)
    probs_arr = np.asarray(probs)
    best_thr, best_f1 = 0.5, -1.0
    for thr in np.linspace(0.05, 0.95, 181):
        preds = (probs_arr >= thr).astype(int)
        f1 = f1_score(labels_arr, preds, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, best_f1

def train_siamese(model, loader, criterion, opt, device):
    model.train()
    total_loss=0; correct=0; total=0
    for q1, q2, labels in loader:
        q1, q2, labels = q1.to(device), q2.to(device), labels.to(device)
        opt.zero_grad()
        logits = model(q1, q2)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += loss.item()
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total += len(labels)
    return total_loss / len(loader), correct / total

def eval_siamese(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss=0; correct=0; total=0
    all_probs=[]; all_labels=[]
    with torch.no_grad():
        for q1, q2, labels in loader:
            q1, q2, labels = q1.to(device), q2.to(device), labels.to(device)
            logits = model(q1, q2)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).long()
            correct += (preds == labels.long()).sum().item()
            total += len(labels)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), correct / total, all_probs, all_labels

history_c={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[], 'val_macro_f1':[], 'val_auc':[], 'threshold':[]}
best_val_macro_f1_c = -1.0
best_threshold_c = 0.5
best_epoch_c = 0
best_model_state_c = None

print("Training Siamese Network...")
print(f"{'Epoch':>5} {'TrainLoss':>10} {'TrainAcc':>9} {'ValLoss':>8} {'ValAcc':>7} {'ValMacroF1':>11} {'ValAUC':>7} {'Thr':>6}")
print("-"*82)
for epoch in range(1, EPOCHS_C+1):
    tr_l, tr_a = train_siamese(model_c, train_c_dl, criterion_c, optimizer_c, DEVICE)
    vl_l, _, val_probs, val_labels = eval_siamese(model_c, val_c_dl, criterion_c, DEVICE, threshold=0.5)
    val_thr, val_macro_f1 = tune_threshold(val_probs, val_labels)
    val_acc = binary_accuracy_from_probs(val_probs, val_labels, threshold=val_thr)
    val_auc = roc_auc_score(val_labels, val_probs) if len(set(int(x) for x in val_labels)) > 1 else 0.0
    scheduler_c.step()
    history_c['train_loss'].append(tr_l);      history_c['val_loss'].append(vl_l)
    history_c['train_acc'].append(tr_a);       history_c['val_acc'].append(val_acc)
    history_c['val_macro_f1'].append(val_macro_f1); history_c['val_auc'].append(val_auc)
    history_c['threshold'].append(val_thr)
    if val_macro_f1 > best_val_macro_f1_c:
        best_val_macro_f1_c = val_macro_f1
        best_threshold_c = val_thr
        best_epoch_c = epoch
        best_model_state_c = {k: v.cpu().clone() for k, v in model_c.state_dict().items()}
    print(f"{epoch:>5} {tr_l:>10.4f} {tr_a:>9.4f} {vl_l:>8.4f} {val_acc:>7.4f} {val_macro_f1:>11.4f} {val_auc:>7.4f} {val_thr:>6.3f}")

if best_model_state_c is not None:
    model_c.load_state_dict(best_model_state_c)
    model_c = model_c.to(DEVICE)

print(f"\nSiamese training complete! Best Val Macro F1: {best_val_macro_f1_c:.4f} at epoch {best_epoch_c} with threshold {best_threshold_c:.3f}")

SiameseNet(
  (encoder): LSTMEncoder(
    (embedding): Embedding(27576, 64, padding_idx=0)
    (lstm): LSTM(64, 128, batch_first=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (projection): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=1, bias=True)
  )
)
Params: 1,905,473
Training Siamese Network...
Epoch  TrainLoss  TrainAcc  ValLoss  ValAcc  ValMacroF1  ValAUC    Thr
----------------------------------------------------------------------------------
    1     0.6494    0.6138   0.6194  0.6660      0.6660  0.7307  0.515
    2     0.6010    0.6759   0.5901  0.6895      0.6894  0.7559  0.460
    3     0.5719    0.7047   0.5870  0.7032      0.7028  0.7708  0.540
    4     0.5436    0.7292   0.5608  0.7162 

In [14]:
# ============================================================
# PART C - Step 4: Evaluation with Accuracy, AUC-ROC & Curves
# ============================================================

from sklearn.metrics import roc_auc_score, roc_curve, f1_score, confusion_matrix

# Test evaluation with the best validation-threshold checkpoint loaded in Cell 13
test_loss_c, _, test_probs_c, test_labels_c = eval_siamese(
    model_c, test_c_dl, criterion_c, DEVICE, threshold=best_threshold_c)

test_preds_c = [int(p >= best_threshold_c) for p in test_probs_c]
test_acc_c = accuracy_score([int(l) for l in test_labels_c], test_preds_c)
test_macro_f1_c = f1_score([int(l) for l in test_labels_c], test_preds_c, average='macro', zero_division=0)
test_auc = roc_auc_score(test_labels_c, test_probs_c)
fpr, tpr, _ = roc_curve(test_labels_c, test_probs_c)
cm_c = confusion_matrix([int(l) for l in test_labels_c], test_preds_c)

print(f"\n{'='*60}")
print(f"PART C RESULTS - Siamese Network")
print(f"{'='*60}")
print(f"Best Val Macro F1: {best_val_macro_f1_c:.4f} (epoch {best_epoch_c})")
print(f"Decision Threshold: {best_threshold_c:.3f}")
print(f"Test Loss:          {test_loss_c:.4f}")
print(f"Test Accuracy:      {test_acc_c:.4f} ({test_acc_c*100:.2f}%)")
print(f"Test Macro F1:      {test_macro_f1_c:.4f}")
print(f"AUC-ROC:            {test_auc:.4f}")
print(f"\nConfusion Matrix [rows=true, cols=pred]:")
print(cm_c)
print(f"\nClassification Report:")
print(classification_report(
    [int(l) for l in test_labels_c],
    test_preds_c,
    target_names=['Not Duplicate', 'Duplicate'],
    zero_division=0))

# Plot training curves + ROC curve
fig3, axes3 = plt.subplots(1, 3, figsize=(18, 5))
fig3.suptitle("Part C: Siamese Network - Evaluation", fontsize=14, fontweight='bold')

epochs_c_range = range(1, EPOCHS_C+1)
axes3[0].plot(epochs_c_range, history_c['train_loss'], 'b-o', label='Train', linewidth=2)
axes3[0].plot(epochs_c_range, history_c['val_loss'],   'r-s', label='Val',   linewidth=2)
axes3[0].set_title('Loss'); axes3[0].set_xlabel('Epoch'); axes3[0].legend(); axes3[0].grid(True, alpha=0.3)

axes3[1].plot(epochs_c_range, history_c['train_acc'], 'b-o', label='Train Acc @0.5', linewidth=2)
axes3[1].plot(epochs_c_range, history_c['val_acc'],   'r-s', label='Val Acc @best thr', linewidth=2)
axes3[1].plot(epochs_c_range, history_c['val_macro_f1'], 'g-^', label='Val Macro F1', linewidth=2)
axes3[1].set_title('Accuracy / Macro F1'); axes3[1].set_xlabel('Epoch'); axes3[1].legend(); axes3[1].grid(True, alpha=0.3)

axes3[2].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC={test_auc:.3f})')
axes3[2].plot([0,1],[0,1],'k--',linewidth=1,label='Random')
axes3[2].set_title('ROC Curve'); axes3[2].set_xlabel('FPR'); axes3[2].set_ylabel('TPR')
axes3[2].legend(); axes3[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/partC_siamese_results.png', dpi=150, bbox_inches='tight')
plt.show()

torch.save({
    'model_state_dict': model_c.state_dict(),
    'threshold': best_threshold_c,
    'vocab': q_word2idx,
    'max_len': MAX_Q_LEN,
    'embed_dim': EMBED_C,
    'hidden_dim': HIDDEN_C,
    'num_layers': LAYERS_C,
}, '/kaggle/working/partC_siamese_net.pth')
print("Siamese checkpoint saved!")
print("\n" + "="*60)
print("ALL 3 PARTS COMPLETE!")
print("="*60)
print(f"Outputs saved in /kaggle/working/:")
print("  partA_lstm_sentiment.pth  - LSTM Sentiment model")
print("  partB_bilstm_ner.pth      - BiLSTM NER model")
print("  partC_siamese_net.pth     - Siamese checkpoint + threshold")
print("  partA_training_curves.png")
print("  partB_ner_curves.png")
print("  partC_siamese_results.png")


PART C RESULTS - Siamese Network
Best Val Macro F1: 0.7251 (epoch 6)
Decision Threshold: 0.505
Test Loss:          0.5462
Test Accuracy:      0.7244 (72.44%)
Test Macro F1:      0.7244
AUC-ROC:            0.8015

Confusion Matrix [rows=true, cols=pred]:
[[1958  824]
 [ 701 2051]]

Classification Report:
               precision    recall  f1-score   support

Not Duplicate       0.74      0.70      0.72      2782
    Duplicate       0.71      0.75      0.73      2752

     accuracy                           0.72      5534
    macro avg       0.72      0.72      0.72      5534
 weighted avg       0.72      0.72      0.72      5534

Siamese checkpoint saved!

ALL 3 PARTS COMPLETE!
Outputs saved in /kaggle/working/:
  partA_lstm_sentiment.pth  - LSTM Sentiment model
  partB_bilstm_ner.pth      - BiLSTM NER model
  partC_siamese_net.pth     - Siamese checkpoint + threshold
  partA_training_curves.png
  partB_ner_curves.png
  partC_siamese_results.png
